# Plant Doctor AI — Experiment B

EfficientNet-B0 + traditional RGB training augmentation (Dataset B).

Only the training transform changes from Experiment A. Validation/test remain deterministic.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

if not Path('/content/drive').exists():
    drive.mount('/content/drive')

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'CUDA/T4 GPU is required.'

In [ ]:
%cd /content/Plant-Doctor-AI
!git pull origin main

In [ ]:
from pathlib import Path
dataset_root = Path('/content/plant_doctor_dataset')
assert all((dataset_root / s).exists() for s in ['train','validation','test']), 'Dataset split missing.'
print('Dataset root:', dataset_root)
for split in ['train','validation','test']:
    count = sum(1 for p in (dataset_root / split).rglob('*') if p.is_file())
    print(f'{split}: {count}')

## Run Experiment B

This uses the exact EfficientNet-B0 trainer as the baseline, but enables the repository's conservative training augmentation. The output is written separately so the A/B ablation remains clean.

In [ ]:
!python -m training.efficientnet_b0_train \
    --dataset-root /content/plant_doctor_dataset \
    --output-dir "/content/drive/MyDrive/Colab Notebooks/Plant-Doctor-AI/results/efficientnet_b0_augmented" \
    --epochs 15 \
    --batch-size 32 \
    --num-workers 2 \
    --lr 3e-4 \
    --weight-decay 1e-4 \
    --early-stop-patience 5 \
    --seed 42 \
    --device cuda \
    --augmented

In [ ]:
import json
import pandas as pd
from pathlib import Path

result_dir = Path('/content/drive/MyDrive/Colab Notebooks/Plant-Doctor-AI/results/efficientnet_b0_augmented')
with open(result_dir / 'metrics.json') as f:
    metrics = json.load(f)
print('Experiment B metrics:')
for k, v in metrics.items():
    print(f'{k}: {v}')
display(pd.read_csv(result_dir / 'classification_report.csv'))
display(pd.read_csv(result_dir / 'confusion_matrix.csv'))